# HolisQA: minimal public-data loader

HolisQA accompanies **CausalRAG2: Hierarchical Causal Knowledge Graph Design for RAG**. It contains multi-sentence question-answer pairs built from open-access scientific-paper abstracts.

This notebook downloads one domain directly from the authors' public GitHub repository and displays two records. No API key or third-party package is required.

Paper: https://arxiv.org/abs/2602.05143  
Dataset: https://github.com/Pwnb/CausalRAG2/tree/main/data/holisqa

In [ ]:
import json
from collections import Counter
from urllib.request import Request, urlopen

DOMAINS = {"biology", "business", "computer_science", "medicine", "psychology"}
DOMAIN = "medicine"  # Change this to load another domain.
assert DOMAIN in DOMAINS

url = (
    "https://raw.githubusercontent.com/Pwnb/CausalRAG2/"
    f"main/data/holisqa/{DOMAIN}/qa.jsonl"
)
request = Request(url, headers={"User-Agent": "holisqa-colab-demo"})
with urlopen(request, timeout=60) as response:
    text = response.read().decode("utf-8")

records = [json.loads(line) for line in text.splitlines() if line.strip()]
print(f"Loaded {len(records):,} records from the {DOMAIN!r} domain")
print(url)

In [ ]:
# Validate the documented core schema. Extra metadata fields are allowed.
required_fields = {
    "qa_id",
    "category",
    "slice_id",
    "question",
    "answer",
    "context_sentence_ids",
    "context_text",
}
for index, record in enumerate(records):
    missing = required_fields - record.keys()
    assert not missing, f"Record {index} is missing: {sorted(missing)}"

print("Schema validated for every loaded record.")
print("Fields in the first record:", sorted(records[0]))

In [ ]:
# Show two complete examples in the dataset's native JSON representation.
for record in records[:2]:
    print(json.dumps(record, indent=2, ensure_ascii=False))
    print("-" * 80)

In [ ]:
# A tiny summary of the selected domain.
print("Categories:", dict(Counter(r["category"] for r in records)))
print("Unique slices:", len({r["slice_id"] for r in records}))
print("Mean evidence sentences per QA:", round(sum(len(r["context_sentence_ids"]) for r in records) / len(records), 2))

## Related public files

Each domain directory also contains `sentences.jsonl`, `articles.jsonl`, `slices.jsonl`, and `all_abstracts.txt`. Replace `qa.jsonl` in the URL above if you want to inspect those source and metadata files.